#COMBINA SEQUENCES + QUERYS -- ALINEAMIENTO MULTIPLE CLUSTAL OMEGA

In [ ]:
import os
import subprocess
from Bio import SeqIO

# === RUTAS ===
ruta_queries = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\identifiacion_patogeno_individual_sequences"
ruta_hits_base = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\Blast\Blastn_02_07_25\Top_Blastn_alignments"
ruta_output = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple"

# Crear carpeta de salida si no existe
os.makedirs(ruta_output, exist_ok=True)

# Procesar todos los archivos sequence_X.fasta
for archivo in os.listdir(ruta_queries):
    if archivo.startswith("sequence_") and archivo.endswith(".fasta"):
        numero = archivo.replace("sequence_", "").replace(".fasta", "")
        query_id = f"Query_{numero}"

        path_query = os.path.join(ruta_queries, archivo)
        path_hits = os.path.join(ruta_hits_base, query_id)

        if not os.path.exists(path_hits):
            print(f"No se encontró carpeta para {query_id}, saltando...")
            continue

        # Crear archivo combinado (query + top hits)
        archivo_combinado = os.path.join(ruta_output, f"{query_id}_combined.fasta")
        with open(archivo_combinado, "w") as salida:
            # Escribir la secuencia original
            for record in SeqIO.parse(path_query, "fasta"):
                SeqIO.write(record, salida, "fasta")
            # Escribir los 5 top hits
            for archivo_hit in os.listdir(path_hits):
                if archivo_hit.endswith(".fasta") or archivo_hit.endswith(".fa"):
                    path_hit_fasta = os.path.join(path_hits, archivo_hit)
                    for record in SeqIO.parse(path_hit_fasta, "fasta"):
                        SeqIO.write(record, salida, "fasta")

        # Archivo alineado
        archivo_alineado = os.path.join(ruta_output, f"{query_id}_alignment.aln")

        # Ejecutar Clustal Omega
        comando = [
            "clustalo",
            "-i", archivo_combinado,
            "-o", archivo_alineado,
            "--outfmt=clustal",
            "--force"
        ]

        try:
            subprocess.run(comando, check=True)
            print(f"✅ Alineamiento completo: {archivo_alineado}")
        except subprocess.CalledProcessError as e:
            print(f"❌ Error al alinear {query_id}: {e}")


✅ Alineamiento completo: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple\Query_1_alignment.aln


CLUSTAL OMEGA

In [1]:
import os
import subprocess
from Bio.Align.Applications import ClustalOmegaCommandline

def run_clustal_omega(input_folder, output_folder):
    """
    Ejecuta Clustal Omega en todos los archivos Combined_[1-13].fasta
    
    Args:
        input_folder: Carpeta con los archivos combinados
        output_folder: Carpeta para los resultados de alineamiento
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Verificar si Clustal Omega está instalado
    try:
        subprocess.run(["clustalo", "--version"], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("Error: Clustal Omega no está instalado o no está en el PATH")
        print("Instala Clustal Omega con: conda install -c bioconda clustalo")
        return
    
    # Procesar cada archivo combinado del 1 al 13
    for seq_num in range(1, 14):
        input_file = f"Combined_{seq_num}.fasta"
        input_path = os.path.join(input_folder, input_file)
        
        if os.path.exists(input_path):
            # Configurar nombres de archivos de salida
            output_file = f"Aligned_{seq_num}.fasta"
            output_path = os.path.join(output_folder, output_file)
            
            # Configurar el comando Clustal Omega
            clustalomega_cline = ClustalOmegaCommandline(
                infile=input_path,
                outfile=output_path,
                verbose=True,
                auto=True,
                seqtype="protein",
                outfmt="fasta"
            )
            
            print(f"\nAlineando {input_file}...")
            print("Comando ejecutado:", clustalomega_cline)
            
            # Ejecutar Clustal Omega
            try:
                stdout, stderr = clustalomega_cline()
                print(f"Alineamiento completado: {output_file}")
            except Exception as e:
                print(f"Error al alinear {input_file}: {str(e)}")
        else:
            print(f"Archivo no encontrado: {input_file}")

if __name__ == "__main__":
    # Configurar rutas (ajustar según tu estructura)
    combined_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\Combinados_query_hits"  # Carpeta con Combined_[1-13].fasta
    alignment_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\alineamiento_multiple"  # Carpeta para resultados
    
    print("=== Ejecutando alineamientos con Clustal Omega ===")
    run_clustal_omega(combined_folder, alignment_folder)
    print("\nProceso de alineamiento completado!")

c:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\.venv\Lib\site-packages\Bio\Application\__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


=== Ejecutando alineamientos con Clustal Omega ===

Alineando Combined_1.fasta...
Comando ejecutado: clustalo -i "C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\Combinados_query_hits\Combined_1.fasta" -t protein -o "C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\alineamiento_multiple\Aligned_1.fasta" --outfmt fasta --auto -v
Alineamiento completado: Aligned_1.fasta

Alineando Combined_2.fasta...
Comando ejecutado: clustalo -i "C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\Combinados_query_hits\Combined_2.fasta" -t pr